# LAM Audio2Expression on Colab

Runs LAM-A2E on a free Colab GPU (T4) instead of locally, since LAM needs CUDA
and this machine only has Intel integrated graphics.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (T4), then Save.

Run the cells in order. LAM only supports Python 3.10, and Colab's default is
newer than that by now, so this builds an isolated 3.10 venv instead of using
conda (the install script itself is plain pip, no conda-specific steps).

In [ ]:
# 1. Confirm the GPU is attached
!nvidia-smi

In [ ]:
# 2. Check what Colab's base OS/Python look like (just informational)
!lsb_release -a
!python3 --version

In [ ]:
# 3. Install Python 3.10 if it's not already present, then create a venv.
# Tries the default apt repos first; falls back to the deadsnakes PPA if
# python3.10 isn't available there (Colab's Ubuntu version changes over time).
!apt-get install -y python3.10 python3.10-venv python3.10-distutils 2>/dev/null \
  || (sudo add-apt-repository -y ppa:deadsnakes/ppa && apt-get update \
      && apt-get install -y python3.10 python3.10-venv python3.10-distutils)

!python3.10 -m venv /content/lam_env
!/content/lam_env/bin/python --version

In [ ]:
# 4. Clone LAM's repo (HTTPS, public repo, no auth needed)
%cd /content
!git clone https://github.com/aigc3d/LAM_Audio2Expression.git
%cd /content/LAM_Audio2Expression

In [ ]:
# 5. Install dependencies into the venv.
# Note: every later step calls the venv's python/pip DIRECTLY by path
# (/content/lam_env/bin/...) instead of `activate`, since each Colab cell
# runs in its own subshell and activation wouldn't persist between cells.
PY = "/content/lam_env/bin/python"
PIP = "/content/lam_env/bin/pip"

!{PIP} install --upgrade pip
# Colab's T4 GPUs run CUDA 12.x drivers, matching install_cu121.sh:
!{PIP} install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -r requirements.txt
!{PIP} install wheels/gradio_gaussian_render-0.0.3-*.whl

In [ ]:
# 6. Download model weights + assets.
# HuggingFace first; if this cell errors (region-blocked etc.), use the OSS
# fallback cell below instead.
PIP = "/content/lam_env/bin/pip"
!{PIP} install -q huggingface_hub
!/content/lam_env/bin/huggingface-cli download 3DAIGC/LAM_audio2exp --local-dir ./
!tar -xzvf LAM_audio2exp_assets.tar && rm -f LAM_audio2exp_assets.tar
!tar -xzvf LAM_audio2exp_streaming.tar && rm -f LAM_audio2exp_streaming.tar

# Sanity check: list what actually landed, since the README doesn't clearly
# state the final folder layout. Adjust the --options weight=... path in the
# inference cell below to match whatever shows up here.
!find . -iname "*.tar" -o -iname "*streaming*" -o -iname "*pretrained*" | head -40

In [ ]:
# 6b. OSS fallback -- only run this if cell 6 failed
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/LAM_audio2exp_assets.tar
!tar -xzvf LAM_audio2exp_assets.tar && rm -f LAM_audio2exp_assets.tar
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/LAM_audio2exp_streaming.tar
!tar -xzvf LAM_audio2exp_streaming.tar && rm -f LAM_audio2exp_streaming.tar
!find . -iname "*.tar" -o -iname "*streaming*" -o -iname "*pretrained*" | head -40

In [ ]:
# 7. Upload the audio you want to run through LAM (one of our clip .wav files)
# Clears out any previously-uploaded .wav files first, so a re-upload of the
# same clip lands as a clean "name.wav" instead of Colab auto-renaming it to
# "name (2).wav" -- that extra space + parenthesis broke inference in a
# couple of different ways before this was added (see cell 8's notes and
# docs/FACIAL_ANIM_REGEN_NOTES.md). Safe to delete -- by this point any
# earlier clip's actual output is already saved under exp/, not here.
import glob
import os

for f in glob.glob("*.wav"):
    os.remove(f)

from google.colab import files
uploaded = files.upload()
wav_name = list(uploaded.keys())[0]
print(f"Uploaded: {wav_name}")

In [ ]:
# 8. Run inference.
# If cell 6's `find` output showed a different weight path than the one
# below, update `weight=` to match before running this.
# save_path is the experiment/log folder (derived from the uploaded filename
# so each clip gets its own folder -- running inference on a second clip
# before downloading the first's result won't silently overwrite it).
# save_json_path is the ACTUAL blendshape output file -- LAM's own README
# sample command omits this (only shows save_path), which looks like a
# documentation bug: the inference code only writes output when
# save_json_path is explicitly set (self.cfg.save_json_path is not None),
# otherwise the computed result is silently discarded. Confirmed by reading
# engines/infer.py directly after a real run produced only infer.log +
# config.py and no data file.
#
# brow_movement=True: enabled 2026-10-23 -- streaming config defaults this
# to False (no brow motion at all); turned on for more expressive output.
# movement_smooth is still untried -- see docs/FACIAL_ANIM_REGEN_NOTES.md
# for the full list of tuning knobs.
#
# No spleeter install here on purpose -- tested it (see
# docs/FACIAL_ANIM_REGEN_NOTES.md), confirmed it makes no measurable
# difference for our clean TTS dialogue clips (nothing to vocal-separate),
# and it's not worth the tensorflow/numpy version risk alongside torch for
# zero benefit. The "spleeter: not found" warning below is expected and
# harmless -- inference.py falls back to the raw audio automatically.
#
# Kept simple on purpose -- a re-uploaded file with the same name used to
# get Colab-renamed to "name (2).wav", and that space + parenthesis broke
# this command outright. Fixed at the source instead (cell 7 now deletes
# old .wav files before every upload, so the rename never happens), rather
# than complicating this cell with shell-quoting/subprocess workarounds.
# See docs/FACIAL_ANIM_REGEN_NOTES.md for what was tried and reverted.
import os
clip_name = os.path.splitext(wav_name)[0]
save_path = f"exp/audio2exp_{clip_name}"
json_path = f"{save_path}/{clip_name}.json"
print(f"Writing output to: {json_path}")

PY = "/content/lam_env/bin/python"
!{PY} inference.py --config-file configs/lam_audio2exp_config_streaming.py --options save_path={save_path} save_json_path={json_path} weight=pretrained_models/lam_audio2exp_streaming.tar audio_input={wav_name} brow_movement=True

In [ ]:
# 9. Find the output JSON and download it. Searches broadly under exp/
# (not just this clip's save_path) so it also works if you're re-running
# this cell standalone without having re-run an updated cell 8 first.
# glob.glob() order is arbitrary (filesystem order, not creation order), so
# this sorts by actual modification time instead of trusting list position.
from google.colab import files
import glob
import os

json_files = glob.glob("exp/**/*.json", recursive=True)
if not json_files:
    raise FileNotFoundError("No JSON found under exp/ -- run !find exp -iname \"*.json\" to check manually, or the recency-based find from troubleshooting if that comes up empty too")

json_files.sort(key=os.path.getmtime)
print(f"Found {len(json_files)} JSON file(s), newest last: {json_files}")
output_path = json_files[-1]
print(f"Downloading (most recently modified): {output_path}")
files.download(output_path)

## Next step

Take the downloaded JSON, put it in `babylon-clay-scene/animations/` in the
VRET-Project repo, then convert it with our own converter:

```powershell
python animations\parse_a2e.py path\to\downloaded.json -o public\animations\my_clip_frames.json
```

See `babylon-clay-scene/ANIMATION_PIPELINE.md` Section 6 for the full writeup.